# Profit Prompts — weekly voiceover batch

Turns a week of reel **plans** into cloned-voice audio plus word-level caption
timings, in the exact shape `reels/src/types.ts` expects.

**Runtime → Change runtime type → T4 GPU** before running anything.

### Why this is a notebook and not a cron job
Colab's terms forbid bypassing the notebook interface and driving it headlessly.
Sitting down once a week and running this by hand is squarely inside the rules;
wiring it into GitHub Actions is not. The rest of the pipeline stays automated —
this one step is deliberately manual.

### What it uses
| Piece | Model | Licence |
|---|---|---|
| Voice cloning | CosyVoice2-0.5B | Apache-2.0 — commercial use OK |
| Caption timings | Whisper | MIT |

Both licences matter: the page is monetised, so the non-commercial models
(XTTS v2, F5-TTS, Fish Speech weights) are off the table regardless of what
hardware they run on.

### Run order
1. GPU check → 2. Install → 3. Load plans → 4. Upload your voice reference →
5. Load model → 6. Generate → 7. Align → 8. Download → commit.

In [ ]:
# 1 — GPU check. Stop here if this says CPU.
import subprocess, sys
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
      or "NO GPU — set Runtime > Change runtime type > T4 GPU, then rerun.")

In [ ]:
# 2 — Install CosyVoice + Whisper. Takes 6-10 minutes; run once per session.
#
# Hard-won notes from the first real run:
#  - Never pipe pip through `tail`. A half-applied requirements.txt prints
#    identically to a clean one and surfaces three cells later as a confusing
#    ModuleNotFoundError.
#  - Never install the tail as one pip command. A resolver conflict on any one
#    package fails the whole command, and packages already present still report
#    "ok", which masks it.
#  - hyperpyyaml resolves classes through pydoc, so missing modules deep in the
#    graph arrive wrapped in ErrorDuringImport rather than ModuleNotFoundError.
#    The resolver below reads the module name out of the exception *text* so it
#    catches both shapes.
%cd /content
!git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git 2>/dev/null || echo "already cloned"
%cd /content/CosyVoice
!git submodule update --init --recursive -q

# Best-effort. Some pins conflict with Colab's preinstalled stack; that is
# expected, which is why the explicit list below exists.
!pip install -r requirements.txt 2>&1 | grep -Ei "error|successfully installed" | tail -15

import importlib, subprocess, sys, re

# Every module CosyVoice reaches for at load time, verified against an actual
# cold run on Colab (Python 3.13). Installed one at a time so a single failure
# cannot take the rest down with it.
TAIL = ["hyperpyyaml", "conformer", "lightning", "diffusers", "onnxruntime",
        "hydra-core", "omegaconf", "pyworld", "inflect", "wget", "soundfile",
        "librosa", "rich", "onnx", "gdown", "modelscope", "openai-whisper",
        "WeTextProcessing"]

for pkg in TAIL:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f"  {pkg}: install failed (continuing)")

sys.path.extend(["/content/CosyVoice", "/content/CosyVoice/third_party/Matcha-TTS"])

# Import for real, and auto-install anything still missing. Module name and pip
# name differ often enough that the mapping is worth keeping.
PIP_NAME = {"hydra": "hydra-core", "tn": "WeTextProcessing",
            "whisper": "openai-whisper", "yaml": "pyyaml",
            "sklearn": "scikit-learn", "PIL": "pillow"}
SKIP = {"matcha", "tn"}   # matcha is a path, tn is optional (see below)

for attempt in range(15):
    try:
        # Drop the half-imported package or Python serves the cached failure.
        for m in [m for m in sys.modules if m.startswith("cosyvoice")]:
            del sys.modules[m]
        importlib.import_module("cosyvoice.cli.cosyvoice")
        print(f"\n✅ cosyvoice imports cleanly (after {attempt} extra install(s))")
        break
    except Exception as e:
        found = re.search(r"No module named ['\"]([\w.]+)['\"]", str(e))
        if not found:
            print(f"\n❌ not a missing-module error:\n{type(e).__name__}: {e}")
            break
        mod = found.group(1).split(".")[0]
        if mod in SKIP:
            print(f"\n❌ cannot auto-fix '{mod}' — see the troubleshooting table")
            break
        pkg = PIP_NAME.get(mod, mod)
        print(f"[{attempt + 1}] missing '{mod}' -> pip install {pkg}")
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                           capture_output=True, text=True)
        if r.returncode != 0:
            print("\n".join((r.stdout + r.stderr).strip().splitlines()[-12:]))
            break
else:
    print("\n❌ still unresolved after 15 attempts")

# tn (WeTextProcessing) normalises numbers and abbreviations. Its pynini
# dependency is the most fragile part of this install and CosyVoice loads
# without it — the workaround is to spell numbers out in the VO script.
try:
    importlib.import_module("tn")
    print("tn (WeTextProcessing)  ok")
except Exception:
    print("tn (WeTextProcessing)  missing — fine. Spell numbers out in `vo`\n"
          "   ('eighty', not '80'). The on-screen beats can still use numerals.")

In [ ]:
# 3 — Config + load this week's plans straight from the repo.
#
# A plan is what the writer produces: {id, vo, beats, handle}. This notebook
# adds audio, captions and duration, and emits the finished reel.json.
import json, os, urllib.request, glob
from pathlib import Path

REPO = "vaibhav018/MemeFactory"
BRANCH = "main"
WORK = Path("/content/work"); WORK.mkdir(exist_ok=True)
(WORK / "audio").mkdir(exist_ok=True)
(WORK / "reels").mkdir(exist_ok=True)

# Names of the plans to voice this week, without the .plan.json suffix.
PLAN_IDS = ["example"]

plans = []
for pid in PLAN_IDS:
    url = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/reels/data/plans/{pid}.plan.json"
    try:
        with urllib.request.urlopen(url) as r:
            plans.append(json.loads(r.read().decode()))
        print(f"  loaded {pid}")
    except Exception as e:
        print(f"  FAILED {pid}: {e}  (upload it manually instead)")

print(f"\n{len(plans)} plan(s) ready")
for p in plans:
    print(f"  {p['id']}: {len(p['vo'].split())} words of VO, {len(p['beats'])} beats")

In [ ]:
# 4 — Upload your voice reference (voice_reference_48k.wav).
#
# Kept out of the repo on purpose: it is your voice, and biometric material
# does not belong in a public repository. Upload it each session.
from google.colab import files

REF = WORK / "reference.wav"
if REF.exists():
    print(f"already uploaded: {REF}")
else:
    up = files.upload()
    name = list(up)[0]
    REF.write_bytes(up[name])
    print(f"saved {name} -> {REF} ({REF.stat().st_size // 1024} KB)")

In [ ]:
# 5 — Load Whisper, transcribe the reference, then load CosyVoice.
#
# CosyVoice's zero-shot mode needs `prompt_text`: what you actually said in the
# reference clip. Transcribing it automatically avoids you having to type it out
# and avoids the mismatch that degrades speaker similarity.
import whisper, torch, torchaudio

asr = whisper.load_model("small")

# 15s is plenty of prompt for zero-shot and keeps conditioning tight.
wav, sr = torchaudio.load(str(REF))
wav = wav.mean(0, keepdim=True)[:, : sr * 15]
CLIP = WORK / "reference_15s.wav"
torchaudio.save(str(CLIP), wav, sr)

PROMPT_TEXT = asr.transcribe(str(CLIP), language="en")["text"].strip()
print(f"reference transcript:\n  {PROMPT_TEXT}\n")

from modelscope import snapshot_download
snapshot_download("iic/CosyVoice2-0.5B", local_dir="pretrained_models/CosyVoice2-0.5B")

from cosyvoice.cli.cosyvoice import CosyVoice2
from cosyvoice.utils.file_utils import load_wav

cosy = CosyVoice2("pretrained_models/CosyVoice2-0.5B",
                  load_jit=False, load_trt=False, fp16=False)
prompt_16k = load_wav(str(CLIP), 16000)
print(f"CosyVoice ready — output sample rate {cosy.sample_rate} Hz")

In [ ]:
# 6 — Generate the voiceovers.
#
# Long scripts are synthesised sentence by sentence and concatenated: it keeps
# prosody stable and means one bad sentence can be rerun without redoing the
# whole take. A short pause is inserted between sentences so captions have
# somewhere to breathe.
import re, torch, torchaudio

PAUSE_S = 0.28

def sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]

for plan in plans:
    parts = []
    for i, sent in enumerate(sentences(plan["vo"]), 1):
        for out in cosy.inference_zero_shot(sent, PROMPT_TEXT, prompt_16k, stream=False):
            parts.append(out["tts_speech"])
        parts.append(torch.zeros(1, int(cosy.sample_rate * PAUSE_S)))
        print(f"  {plan['id']}  sentence {i} ok")
    audio = torch.cat(parts, dim=1)
    dest = WORK / "audio" / f"{plan['id']}.wav"
    torchaudio.save(str(dest), audio, cosy.sample_rate)
    plan["_audio"] = dest
    plan["_seconds"] = audio.shape[1] / cosy.sample_rate
    print(f"{plan['id']}: {plan['_seconds']:.1f}s -> {dest.name}\n")

In [ ]:
# 7 — Word-level alignment, encode to MP3, and write the finished reel.json.
#
# Whisper runs over the *generated* audio rather than the script, so the timings
# describe what was actually said. Captions drift otherwise.
#
# MP3 rather than WAV because the render job in CI needs this audio: a 45s mono
# VO is ~360 KB as 64k MP3 against ~8 MB as WAV, which is the difference between
# a repo that stays clonable and one that does not.
import subprocess

for plan in plans:
    res = asr.transcribe(str(plan["_audio"]), language="en", word_timestamps=True)
    captions = [
        {"word": w["word"].strip(), "start": round(w["start"], 3), "end": round(w["end"], 3)}
        for seg in res["segments"] for w in seg.get("words", [])
        if w["word"].strip()
    ]

    mp3 = plan["_audio"].with_suffix(".mp3")
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", str(plan["_audio"]),
         "-ac", "1", "-b:a", "64k", str(mp3)],
        check=True,
    )
    plan["_mp3"] = mp3

    reel = {
        "id": plan["id"],
        "audioSrc": f"audio/{plan['id']}.mp3",
        "durationInSeconds": round(plan["_seconds"], 2),
        "handle": plan.get("handle", "@profit_prompts_"),
        "brollSrc": None,
        "captions": captions,
        "beats": plan["beats"],
    }
    (WORK / "reels" / f"{plan['id']}.reel.json").write_text(
        json.dumps(reel, indent=2, ensure_ascii=False), encoding="utf-8")

    last_beat = max(b["at"] for b in plan["beats"])
    print(f"{plan['id']}: {len(captions)} words, {reel['durationInSeconds']}s, "
          f"mp3 {mp3.stat().st_size // 1024} KB")
    if last_beat > reel["durationInSeconds"]:
        print(f"   WARNING last beat at {last_beat}s is past the end of the audio —"
              f" retime the beats in the plan")
    elif last_beat < reel["durationInSeconds"] - 6:
        print(f"   NOTE last beat at {last_beat}s, audio runs {reel['durationInSeconds']}s"
              f" — {reel['durationInSeconds'] - last_beat:.1f}s with no card on screen")

In [ ]:
# 8 — Listen before you ship. Bad takes are obvious and cheap to redo here.
from IPython.display import Audio, display
for plan in plans:
    print(plan["id"])
    display(Audio(str(plan["_audio"])))

In [ ]:
# 9 — Package the MP3s and the reel.json files, and download.
#
# Unpack into the repo so the layout matches what Remotion expects:
#   reels/public/audio/<id>.mp3
#   reels/data/<id>.reel.json
# Both get committed. The intermediate WAVs stay here in Colab — they are large
# and the MP3 is what the render actually reads.
import shutil
from google.colab import files

out = Path("/content/voiceover_batch")
if out.exists():
    shutil.rmtree(out)
(out / "public" / "audio").mkdir(parents=True)
(out / "data").mkdir(parents=True)

for plan in plans:
    shutil.copy(plan["_mp3"], out / "public" / "audio" / plan["_mp3"].name)
    shutil.copy(WORK / "reels" / f"{plan['id']}.reel.json",
                out / "data" / f"{plan['id']}.reel.json")

archive = shutil.make_archive("/content/voiceover_batch", "zip", out)
print(f"{archive}  ({os.path.getsize(archive) // 1024} KB)")
for f in sorted(out.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(out)}  {f.stat().st_size // 1024} KB")
files.download(archive)

## After downloading

```bash
unzip voiceover_batch.zip -d reels/
cd reels
npx remotion render Reel out/<id>.mp4 --props=data/<id>.reel.json
```

Watch it once, then commit **both** the `.mp3` and the `.reel.json` — the render
job in CI needs the audio. `.gitignore` allows `reels/public/audio/*.mp3` and
blocks `*.wav`, which is also what keeps your voice reference out of the repo.

Once committed, `profit_prompts_reel.yml` picks it up at 13:00 UTC (18:30 IST).

### Weekly rhythm

| When | What |
|---|---|
| Sunday | Write next week's plans, run this notebook, commit the batch |
| Daily | Cron renders and posts from the committed `reel.json` |

Scripts have to exist a few days ahead — that is the one thing this free path
costs you against a paid TTS API. Something genuinely time-sensitive can still
go out as a carousel the same day.

### If something goes wrong

| Symptom | Fix |
|---|---|
| `ModuleNotFoundError: hyperpyyaml` (or conformer, diffusers, lightning) | `requirements.txt` half-applied. Rerun cell 2 — it installs these explicitly and verifies them |
| `ModuleNotFoundError: matcha` | Submodule missing — rerun cell 2, it sets `sys.path` |
| `ModuleNotFoundError: tn` | WeTextProcessing/pynini failed. Not fatal: spell numbers out in the VO script (`eighty`, not `80`) |
| Voice sounds unlike you | Reference clip noisy or under ~10s. Use a cleaner 15s window |
| Captions drift from the audio | Whisper must run on `plan["_audio"]`, not the script text |
| Beats land at the wrong moment | Retime `at` in the plan; cell 7 warns on the obvious cases |
| `ffmpeg: not found` | `!apt-get install -y ffmpeg` |
| CUDA out of memory | Restart runtime. 0.5B fits a T4 easily, so this means a stale session |

### A rule this notebook learned the hard way

Never pipe `pip install` through `tail` or `grep` in a way that can hide a
failure. A silent partial install is indistinguishable from a good one until it
surfaces several cells later as a confusing import error.